## Learning notes — Py12. Transform Refunds Data

### Learning objectives
Follow **Py12. Transform Refunds Data** from its source inputs through its operations to the archived results.
- Explain schema, profiling as used in this lesson.

### Source-specific learning focus
- Compare raw fields with typed records and parsing failures.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.

SQL relation identifiers referenced in source (inputs and/or destinations; not a claim that these objects exist locally): `gizmobox.bronze.py_refunds`, `gizmobox.silver.py_refunds`.


## Learning notes — supplied input snapshot

Path: `data/gizmobox/refunds/refunds_azure_sql_ddl.sql`

SHA-256: `381b2dd0af7030cf52c46a562645ca7177b1cdbadea55cb453577694a1949feb`

Complete supplied text (not generated output):

```sql
CREATE TABLE Refunds (  
    refund_id INT PRIMARY KEY,  
    payment_id INT NOT NULL,  
    refund_timestamp DATETIME NOT NULL,  
    refund_amount DECIMAL(10, 2) NOT NULL,  
    refund_reason NVARCHAR(255) NOT NULL. 
);

INSERT INTO Refunds (refund_id, payment_id, refund_timestamp, refund_amount, refund_reason)  
VALUES  
(1, 66, '2025-01-10 11:30:00', 85.75, 'Payment Error:Retailer'),  
(2, 69, '2025-01-03 12:40:15', 120.50, 'Order Cancelled:Customer'),  
(3, 72, '2025-01-06 14:45:30', 65.00, 'Product Returned:Customer'),  
(4, 73, '2025-01-07 16:10:45', 210.99, 'Order Cancelled:Customer'),  
(5, 75, '2025-01-09 18:25:00', 45.20, 'Payment Error:Retailer'),  
(6, 80, '2025-01-10 09:35:20', 130.15, 'Order Cancelled:Customer'),  
(7, 83, '2025-01-12 11:20:40', 150.00, 'Product Returned:Customer'),  
(8, 85, '2025-01-14 13:15:30', 89.99, 'Order Cancelled:Customer'),  
(9, 89, '2025-01-15 15:00:00', 78.50, 'Payment Error:Retailer'),  
(10, 91, '2025-01-17 16:45:15', 250.75, 'Product Returned:Customer');
```

## Learning notes — corrective commentary (originals unchanged)

Refunds setup SQL contains a NOT NULL. typo: the period must be removed in a separate corrected copy before database setup. The supplied SQL is setup, not remote Refunds table contents.

# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Transform Refunds Data
1. Extract specific portion of the string from refund_reason using split function
1. Extract specific portion of the string from refund_reason using regexp_extract function
1. Extract date and time from the refund_timestamp
1. Write transformed data to the Silver schema

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df_refunds = spark.table('gizmobox.bronze.py_refunds')
display(df_refunds)

Archived table:
refund_id	payment_id	refund_timestamp	refund_amount	refund_reason
1	66	2025-01-10T11:30:00Z	85.75	Payment Error:Retailer
2	69	2025-01-03T12:40:15Z	120.50	Order Cancelled:Customer
3	72	2025-01-06T14:45:30Z	65.00	Product Returned:Customer
4	73	2025-01-07T16:10:45Z	210.99	Order Cancelled:Customer
5	75	2025-01-09T18:25:00Z	45.20	Payment Error:Retailer
6	80	2025-01-10T09:35:20Z	130.15	Order Cancelled:Customer
7	83	2025-01-12T11:20:40Z	150.00	Product Returned:Customer
8	85	2025-01-14T13:15:30Z	89.99	Order Cancelled:Customer
9	89	2025-01-15T15:00:00Z	78.50	Payment Error:Retailer
10	91	2025-01-17T16:45:15Z	250.75	Product Returned:Customer

### 1. Extract specific portion of the string from refund_reason using split function

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
from pyspark.sql import functions as F
df_split_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            "refund_timestamp",
            "refund_amount",
            F.split("refund_reason", ":")[0].alias("refund_reason"),
            F.split("refund_reason", ":")[1].alias("refund_source")
        )
)
display(df_split_refunds)

Archived table:
refund_id	payment_id	refund_timestamp	refund_amount	refund_reason	refund_source
1	66	2025-01-10T11:30:00Z	85.75	Payment Error	Retailer
2	69	2025-01-03T12:40:15Z	120.50	Order Cancelled	Customer
3	72	2025-01-06T14:45:30Z	65.00	Product Returned	Customer
4	73	2025-01-07T16:10:45Z	210.99	Order Cancelled	Customer
5	75	2025-01-09T18:25:00Z	45.20	Payment Error	Retailer
6	80	2025-01-10T09:35:20Z	130.15	Order Cancelled	Customer
7	83	2025-01-12T11:20:40Z	150.00	Product Returned	Customer
8	85	2025-01-14T13:15:30Z	89.99	Order Cancelled	Customer
9	89	2025-01-15T15:00:00Z	78.50	Payment Error	Retailer
10	91	2025-01-17T16:45:15Z	250.75	Product Returned	Customer

### 2. Extract specific portion of the string from refund_reason using regexp_extract function

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df_transformed_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            F.date_format('refund_timestamp','yyyy-MM-dd').cast('date').alias('refund_date'),
            F.date_format('refund_timestamp','HH-mm-ss').alias('refund_time'),
            "refund_amount",
            F.regexp_extract("refund_reason", "^([^:]+):", 1).alias("refund_reason"),
            F.regexp_extract("refund_reason", "^[^:]+:(.*)$", 1).alias("refund_source")
        )
)
display(df_transformed_refunds)

Archived table:
refund_id	payment_id	refund_date	refund_time	refund_amount	refund_reason	refund_source
1	66	2025-01-10	11-30-00	85.75	Payment Error	Retailer
2	69	2025-01-03	12-40-15	120.50	Order Cancelled	Customer
3	72	2025-01-06	14-45-30	65.00	Product Returned	Customer
4	73	2025-01-07	16-10-45	210.99	Order Cancelled	Customer
5	75	2025-01-09	18-25-00	45.20	Payment Error	Retailer
6	80	2025-01-10	09-35-20	130.15	Order Cancelled	Customer
7	83	2025-01-12	11-20-40	150.00	Product Returned	Customer
8	85	2025-01-14	13-15-30	89.99	Order Cancelled	Customer
9	89	2025-01-15	15-00-00	78.50	Payment Error	Retailer
10	91	2025-01-17	16-45-15	250.75	Product Returned	Customer

### 3. Write transformed data to the Silver schema  

In [ ]:
df_transformed_refunds.writeTo("gizmobox.silver.py_refunds").createOrReplace()

Archived result list: empty.

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
%sql
SELECT * FROM gizmobox.silver.py_refunds;
```

In [ ]:
# Archived output only; no computation.

Archived table:
refund_id	payment_id	refund_date	refund_time	refund_amount	refund_reason	refund_source
1	66	2025-01-10	11-30-00	85.75	Payment Error	Retailer
2	69	2025-01-03	12-40-15	120.50	Order Cancelled	Customer
3	72	2025-01-06	14-45-30	65.00	Product Returned	Customer
4	73	2025-01-07	16-10-45	210.99	Order Cancelled	Customer
5	75	2025-01-09	18-25-00	45.20	Payment Error	Retailer
6	80	2025-01-10	09-35-20	130.15	Order Cancelled	Customer
7	83	2025-01-12	11-20-40	150.00	Product Returned	Customer
8	85	2025-01-14	13-15-30	89.99	Order Cancelled	Customer
9	89	2025-01-15	15-00-00	78.50	Payment Error	Retailer
10	91	2025-01-17	16-45-15	250.75	Product Returned	Customer